<a href="https://colab.research.google.com/github/bercyx27/PYTHON-BIOLOGY-AND-CHEMISTRY/blob/main/Project_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Project 1
Procedural Programming

In [12]:
# 1. Install required packages
!pip install -q pandas requests

import requests
import pandas as pd
from datetime import date, datetime, timedelta
from typing import Optional
from IPython.display import HTML, display

# ==========================================
# PHASE 1: DATA ACQUISITION
# ==========================================

def validate_date(date_str: str, date_type: str):
    """Validates the format of a date to comply with YYYY-MM-DD."""
    try:
        datetime.strptime(date_str, "%Y-%m-%d")
    except ValueError:
        raise ValueError(
            f"The date format supplied for `{date_type}` is wrong. "
            "Please use YYYY-MM-DD."
        )

# UPDATED: Removed dwd_station_id, added lat and lon (defaults to Stuttgart)
def get_weather_data(since_date: str, end_date: Optional[str] = None, lat: float = 48.7758, lon: float = 9.1829) -> pd.DataFrame:
    """Fetches historical weather data from the Bright Sky API using coordinates."""
    validate_date(since_date, "since_date")

    if not end_date:
        end_date = date.today().strftime("%Y-%m-%d")
    else:
        validate_date(end_date, "end_date")

    url = "https://api.brightsky.dev/weather"

    # UPDATED: Passing lat/lon instead of a station ID
    params = {
        "date": since_date,
        "last_date": end_date,
        "lat": lat,
        "lon": lon,
    }

    headers = {"Accept": "application/json"}

    print(f"Fetching weather data for Lat: {lat}, Lon: {lon} from {since_date} to {end_date}...")
    response = requests.get(url, headers=headers, params=params, timeout=10)
    response.raise_for_status()

    df = pd.DataFrame(response.json()["weather"])
    df["timestamp"] = pd.to_datetime(df["timestamp"])

    return df

# ==========================================
# PHASE 2: DATA PROCESSING
# ==========================================

def process_data_for_ui(df: pd.DataFrame):
    """Extracts UI-ready metrics from the raw Bright Sky DataFrame."""
    df = df.dropna(subset=['temperature'])

    current_row = df.iloc[-1]
    current_temp = current_row['temperature']
    current_icon = current_row['icon']

    hourly_df = df.tail(5).copy()
    hourly_df['time_str'] = hourly_df['timestamp'].dt.strftime('%H:%00')

    df['date_only'] = df['timestamp'].dt.date
    daily_df = df.groupby('date_only').agg(
        temp_max=('temperature', 'max'),
        temp_min=('temperature', 'min'),
        precip_max=('precipitation', 'max'),
        icon=('icon', 'first')
    ).reset_index().tail(5)

    return current_temp, current_icon, hourly_df, daily_df

def get_svg_for_brightsky(icon_str: str):
    """Maps Bright Sky icon strings to clean SVG vectors."""
    if "night" in str(icon_str):
        return '''<svg width="28" height="28" viewBox="0 0 24 24" fill="none"><path d="M21 12.75C21 17.8576 16.8576 22 11.75 22C8.3073 22 5.31969 20.1085 3.7915 17.2915C6.0123 17.8488 8.4414 17.3486 10.2319 15.5581C12.0224 13.7676 12.5226 11.3385 11.9653 9.1177C14.7823 7.58951 16.6738 4.6019 16.6738 1.16C21.7814 1.16 25.9238 5.3024 25.9238 10.41" fill="#f1c40f"/></svg>'''
    elif "cloud" in str(icon_str) or "fog" in str(icon_str):
        return '''<svg width="28" height="28" viewBox="0 0 24 24" fill="none"><path d="M19.384 13.5a4.5 4.5 0 0 0-8.634-1.5A5.5 5.5 0 1 0 12 23h7.384a3.5 3.5 0 0 0 0-7z" fill="#ecf0f1"/></svg>'''
    elif "rain" in str(icon_str):
        return '''<svg width="28" height="28" viewBox="0 0 24 24" fill="none"><path d="M19.384 13.5a4.5 4.5 0 0 0-8.634-1.5A5.5 5.5 0 1 0 12 23h7.384a3.5 3.5 0 0 0 0-7z" fill="#b2bec3"/><path d="M12 23v2M8 22v2M16 22v2" stroke="#74b9ff" stroke-width="2"/></svg>'''
    else:
        return '''<svg width="28" height="28" viewBox="0 0 24 24" fill="none"><circle cx="12" cy="12" r="5" fill="#f39c12"/><path d="M12 1v3M12 20v3M4.22 4.22l2.12 2.12M17.66 17.66l2.12 2.12M1 12h3M20 12h3M4.22 19.78l2.12-2.12M17.66 6.34l2.12-2.12" stroke="#f39c12" stroke-width="2" stroke-linecap="round"/></svg>'''

# ==========================================
# PHASE 3: UI RENDERING
# ==========================================

def render_dashboard(city_name: str, current_temp: float, current_icon: str, hourly_df: pd.DataFrame, daily_df: pd.DataFrame):
    """Compiles the processed Pandas data into the premium HTML interface."""

    hourly_html = ""
    for _, row in hourly_df.iterrows():
        t_str = row['time_str']
        temp = round(row['temperature'])
        icon_svg = get_svg_for_brightsky(row['icon'])
        precip = round(row['precipitation'] * 10) if pd.notna(row['precipitation']) else 0

        hourly_html += f'''
        <div style="text-align:center; flex:1">
            <div style="font-size:13px; opacity:0.8">{t_str}</div>
            {icon_svg}
            <div style="font-weight:bold; margin:5px 0">{temp}°C</div>
            <div style="font-size:11px; color:#74b9ff">💧 {precip}%</div>
        </div>'''

    daily_html = ""
    for _, row in daily_df.iterrows():
        day_name = row['date_only'].strftime("%a")
        max_t = round(row['temp_max'])
        min_t = round(row['temp_min'])
        icon_svg = get_svg_for_brightsky(row['icon'])
        precip = round(row['precip_max'] * 10) if pd.notna(row['precip_max']) else 0

        total_range = 35 - (-5)
        left_pct = ((min_t - (-5)) / total_range) * 100
        width_pct = ((max_t - min_t) / total_range) * 100

        daily_html += f'''
        <div style="display:flex; align-items:center; margin-bottom:12px">
            <div style="width:45px; font-weight:500">{day_name}</div>
            <div style="width:35px">{icon_svg}</div>
            <div style="flex:1; display:flex; align-items:center; gap:8px">
                <span style="width:30px; font-size:12px">{min_t}°</span>
                <div style="flex:1; height:6px; background:rgba(255,255,255,0.1); border-radius:3px; position:relative">
                    <div style="position:absolute; height:100%; left:{left_pct}%; width:{width_pct}%; background:linear-gradient(90deg, #74b9ff, #f39c12); border-radius:3px"></div>
                </div>
                <span style="width:30px; font-size:12px; text-align:right">{max_t}°</span>
            </div>
            <div style="width:45px; text-align:right; font-size:12px; color:#74b9ff">💧 {precip}%</div>
        </div>'''

    html = f'''
    <div style="max-width: 440px; margin: 20px auto; background: #1a2536; border-radius: 30px; padding: 24px; color: white; font-family: sans-serif; box-shadow: 0 10px 30px rgba(0,0,0,0.5);">
        <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:15px">
            <h1 style="margin:0; font-size:28px">{city_name}</h1>
            <div style="background:rgba(255,255,255,0.1); border-radius:50%; padding:5px 10px; cursor:pointer">•••</div>
        </div>
        <div style="display:flex; justify-content:space-between; margin-bottom:20px">
            <h2 style="margin:0; font-size:82px; font-weight:200">{round(current_temp)}°C</h2>
            {get_svg_for_brightsky(current_icon)}
        </div>

        <div style="background:rgba(255,255,255,0.06); padding:15px; border-radius:20px; margin-bottom:15px">
            <h3 style="margin:0 0 15px 0; font-size:13px; color:rgba(255,255,255,0.6)">RECENT HOURLY ACTIVITY</h3>
            <div style="display:flex; justify-content:space-between">{hourly_html}</div>
        </div>

        <div style="background:rgba(255,255,255,0.06); padding:15px; border-radius:20px; margin-bottom:15px">
            <h3 style="margin:0 0 15px 0; font-size:13px; color:rgba(255,255,255,0.6)">5-DAY HISTORICAL DATA</h3>
            {daily_html}
        </div>
    </div>'''

    display(HTML(html))

# ==========================================
# MAIN EXECUTION SCRIPT
# ==========================================
if __name__ == "__main__":
    end_dt = date.today().strftime("%Y-%m-%d")
    start_dt = (date.today() - timedelta(days=5)).strftime("%Y-%m-%d")

    # UPDATED: Passing Stuttgart coordinates directly. No station ID!
    raw_df = get_weather_data(since_date=start_dt, end_date=end_dt, lat=48.7758, lon=9.1829)

    c_temp, c_icon, h_df, d_df = process_data_for_ui(raw_df)

    # UPDATED: Changed city name to Stuttgart
    render_dashboard(city_name="Stuttgart", current_temp=c_temp, current_icon=c_icon, hourly_df=h_df, daily_df=d_df)

Fetching weather data for Lat: 48.7758, Lon: 9.1829 from 2026-06-22 to 2026-06-27...


# New Section